# NLU HOSTAGE — ai_1_nlu_v3_300

Notebook ini hanya memakai modul training bersama. Semua cell legacy telah dihapus.


In [1]:
from pathlib import Path
DATASET_FILENAME = "v3_chat_dataset2_300.csv"
NOTEBOOK_FOLDER = "ai_1_nlu_v3_300"
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != NOTEBOOK_FOLDER:
    candidate = NOTEBOOK_DIR / NOTEBOOK_FOLDER
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate
DATASET_PATH = NOTEBOOK_DIR / "data" / DATASET_FILENAME
print(f"Dataset aktif: {DATASET_PATH}")


Dataset aktif: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v3_300\data\v3_chat_dataset2_300.csv


In [2]:
# MODUL BERSAMA: EDA + SVM + SVM TUNING + NB + NB TUNING + TRANSFORMER
from pathlib import Path
import sys

PROJECT_ROOT = Path(NOTEBOOK_DIR).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.nlu_eda import run_nlu_eda
from modules.nlu_training import (
    predict_intent as _predict_intent_shared,
    predict_transformer_intent as _predict_transformer_intent_shared,
    run_all_nlu_models,
    train_naive_bayes,
    train_naive_bayes_tuned,
    train_svm,
    train_svm_tuned,
    train_transformer,
)

MODEL_DIR = Path(NOTEBOOK_DIR) / "models"
# Kebijakan proyek: SVM/NB pada CPU, Transformer pada GPU CUDA.
CLASSICAL_DEVICE = "cpu"
TRANSFORMER_DEVICE = "cuda"


def run_eda(plot=True):
    return run_nlu_eda(DATASET_PATH, plot=plot)

def train_svm_model():
    return train_svm(DATASET_PATH, MODEL_DIR)

def train_svm_tuned_model():
    return train_svm_tuned(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_model():
    return train_naive_bayes(DATASET_PATH, MODEL_DIR)

def train_naive_bayes_tuned_model():
    return train_naive_bayes_tuned(DATASET_PATH, MODEL_DIR)

def train_transformer_model(epochs=4):
    return train_transformer(DATASET_PATH, MODEL_DIR, epochs=epochs, device=TRANSFORMER_DEVICE)

def predict_intent(text, model_filename=None):
    return _predict_intent_shared(text, MODEL_DIR, model_filename)

def predict_transformer_intent(text):
    return _predict_transformer_intent_shared(text, MODEL_DIR, device=TRANSFORMER_DEVICE)

HOSTAGE_TEST_CASES = [
    ("accusing", "B kena Gag Order saat menjelaskan alibi, menurut gw itu pola Hitman."),
    ("defending", "Gw bukan Hitman, tuduhan itu gak punya bukti publik."),
    ("bluffing", "Gw Spy, semalam gw Guard Raka dan dia pasti aman."),
    ("probing", "Stalker, semalam lu Peek siapa dan hasilnya apa?"),
    ("deflecting", "Jangan fokus ke gw, cek D yang terus mengubah cerita tiap ditanya."),
    ("persuading", "Vote C aja, dia paling diuntungkan dari korban Hostage semalam."),
    ("claiming", "Klaim gw Civilian, gw gak punya skill malam."),
    ("neutral", "Fase malam bikin chat terkunci, kita tunggu pagi dulu."),
]

def run_hostage_test_suite(model_filename=None):
    correct = 0
    for expected, chat in HOSTAGE_TEST_CASES:
        predicted, confidence = predict_intent(chat, model_filename)
        correct += predicted == expected
        print(f"{expected:12} | prediksi={predicted:12} | confidence={confidence:6.2f}% | {chat}")
    print(f"\nCocok: {correct}/{len(HOSTAGE_TEST_CASES)}")

print("Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),")
print("train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().")
print("Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.")
print("Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.")


Modul NLU siap. Jalankan: run_eda(), train_svm_model(), train_svm_tuned_model(),
train_naive_bayes_model(), train_naive_bayes_tuned_model(), atau train_transformer_model().
Mode training aktif: CPU untuk SVM/Naive Bayes, GPU CUDA untuk Transformer.
Model tersimpan terpisah; prediksi default memprioritaskan SVM tuned.


In [3]:
# JALANKAN SEMUA MODEL: empat model CPU, lalu Transformer CUDA dan 10 chat uji.
RUN_TRANSFORMER = True
RUN_TUNING = False
TRANSFORMER_EPOCHS = 4
artifacts, hasil_training, hasil_manual_test = run_all_nlu_models(
    DATASET_PATH, MODEL_DIR,
    run_transformer=RUN_TRANSFORMER,
    run_tuning=RUN_TUNING,
    transformer_epochs=TRANSFORMER_EPOCHS,
    transformer_device=TRANSFORMER_DEVICE,
)
print('RINGKASAN EVALUASI HOLDOUT:')
display(hasil_training)
print('RINGKASAN 10 CHAT UJI:')
display(hasil_manual_test)



MENJALANKAN: SVM baseline
SVM | train=2224 | test=557 | kelas=8



--- Evaluasi SVM baseline (holdout test set) ---
Accuracy    : 0.7343
Macro F1    : 0.7359
Weighted F1 : 0.7319
              precision    recall  f1-score   support

    accusing       0.72      0.75      0.73        72
    bluffing       0.60      0.64      0.62        72
    claiming       0.68      0.67      0.68        73
   defending       0.68      0.60      0.64        72
  deflecting       0.79      0.74      0.77        66
     neutral       0.90      0.91      0.91        70
  persuading       0.72      0.64      0.68        72
     probing       0.79      0.97      0.87        60

    accuracy                           0.73       557
   macro avg       0.74      0.74      0.74       557
weighted avg       0.73      0.73      0.73       557

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v3_300\models\intent_classifier_svm.pkl

MENJALANKAN: Naive Bayes baseline
Naive Bayes | train=2224 | test=557 | kelas=8

--- Evaluasi Naive Bayes baseline 

Model tersimpan: C:\Users\andyc\Documents\a_skripsi\training\prethesis\ai_1_nlu_v3_300\models\intent_classifier_nb.pkl

MENJALANKAN: IndoBERT Transformer


C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Map:   0%|          | 0/2224 [00:00<?, ? examples/s]

...

expected=accusing     | predicted=deflecting   | confidence= 38.70% | MISS
expected=defending    | predicted=neutral      | confidence= 30.07% | MISS

--- 10 chat uji: IndoBERT Transformer ---


expected=accusing     | predicted=accusing     | confidence= 97.40% | OK
expected=defending    | predicted=defending    | confidence= 97.44% | OK
expected=bluffing     | predicted=bluffing     | confidence= 54.23% | OK
expected=probing      | predicted=probing      | confidence= 98.83% | OK
expected=deflecting   | predicted=deflecting   | confidence= 45.90% | OK
expected=persuading   | predicted=accusing     | confidence= 53.48% | MISS
expected=claiming     | predicted=defending    | confidence= 59.71% | MISS
expected=neutral      | predicted=neutral      | confidence= 98.80% | OK
expected=accusing     | predicted=accusing     | confidence= 93.30% | OK
expected=defending    | predicted=defending    | confidence= 97.59% | OK
RINGKASAN EVALUASI HOLDOUT:


,model,accuracy_holdout,macro_f1_holdout,weighted_f1_holdout,waktu_detik,status
0,IndoBERT Transformer,0.8097,0.8129,0.8091,76.6,berhasil
1,SVM baseline,0.7343,0.7359,0.7319,0.5,berhasil
2,Naive Bayes baseline,0.6948,0.7020,0.6969,0.2,berhasil


RINGKASAN 10 CHAT UJI:


,model,benar_dari_10,akurasi_10_chat
0,IndoBERT Transformer,8,0.8
1,SVM baseline,7,0.7
2,Naive Bayes baseline,6,0.6


## Laporan eksekusi notebook

Tuning SVM dan Naive Bayes dilewati untuk mempercepat run ini. Output training lengkap tersimpan pada cell tepat di atas.

| Model | Macro-F1 holdout | Uji 10 chat |
|---|---:|---:|
| SVM baseline | 0.7359 | 7/10 |
| Naive Bayes baseline | 0.7020 | 6/10 |
| IndoBERT Transformer (GPU) | 0.8129 | 8/10 |

Transformer menjadi pilihan terbaik untuk v3_300 karena unggul di holdout dan 10 chat manual.